In [2]:
# src/data_generator/generate_all_data.py

import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

# ============================================================================
# CONFIGURATION
# ============================================================================

fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Database connection
DB_USER = os.getenv("DB_USER", "med_analyst")
DB_PASSWORD = os.getenv("DB_PASSWORD", "your_secure_password")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "medical_practice")
engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Reference data
INSURANCE_PROVIDERS = [
    ("Blue Cross Blue Shield", "PPO"),
    ("Blue Cross Blue Shield", "HMO"),
    ("Aetna", "PPO"),
    ("Aetna", "HMO"),
    ("United Healthcare", "POS"),
    ("Cigna", "EPO"),
    ("Humana", "PPO"),
    ("Medicare", "Part B"),
    ("Medicaid", "Managed Care"),
    ("Kaiser Permanente", "HMO")
]

SPECIALTIES = [
    "Family Medicine", "Internal Medicine", "Cardiology", 
    "Orthopedics", "Pediatrics", "Dermatology", "Neurology",
    "Psychiatry", "OB/GYN", "Endocrinology"
]

# CPT codes with realistic distributions
CPT_CODES = [
    # (code, description, avg_charge, probability_weight)
    ("99213", "Office visit established patient level 3", 75.00, 25),
    ("99214", "Office visit established patient level 4", 110.00, 20),
    ("99203", "Office visit new patient level 3", 120.00, 10),
    ("99204", "Office visit new patient level 4", 170.00, 5),
    ("97110", "Therapeutic exercises", 55.00, 8),
    ("97140", "Manual therapy techniques", 65.00, 6),
    ("85025", "Complete blood count", 25.00, 15),
    ("71045", "Chest X-ray single view", 90.00, 5),
    ("93000", "Electrocardiogram complete", 85.00, 7),
    ("80053", "Comprehensive metabolic panel", 30.00, 10),
    ("81003", "Urinalysis automated with microscopy", 15.00, 8),
    ("90471", "Immunization administration", 40.00, 12),
    ("G0008", "Admin influenza virus vaccine", 35.00, 10),
]

# ICD-10 codes by specialty
ICD10_BY_SPECIALTY = {
    "Family Medicine": [
        ("I10", "Essential hypertension", 20),
        ("E11.9", "Type 2 diabetes mellitus", 15),
        ("E78.5", "Hyperlipidemia", 12),
        ("J06.9", "Acute upper respiratory infection", 10),
        ("M54.5", "Low back pain", 8),
        ("J45.909", "Unspecified asthma", 6),
        ("F41.9", "Anxiety disorder", 5),
        ("Z00.00", "General adult medical exam", 15),
        ("R05", "Cough", 7),
        ("R10.9", "Abdominal pain", 5),
    ],
    "Cardiology": [
        ("I10", "Essential hypertension", 30),
        ("I25.10", "Atherosclerotic heart disease", 20),
        ("I48.91", "Atrial fibrillation", 10),
        ("R07.9", "Chest pain unspecified", 15),
        ("I50.9", "Heart failure unspecified", 10),
        ("E78.5", "Hyperlipidemia", 15),
    ],
    "Pediatrics": [
        ("J06.9", "Acute upper respiratory infection", 25),
        ("Z00.129", "Well-child visit", 30),
        ("H66.9", "Otitis media", 15),
        ("J45.909", "Unspecified asthma", 10),
        ("R05", "Cough", 10),
        ("B34.9", "Viral infection", 10),
    ],
    "Orthopedics": [
        ("M54.5", "Low back pain", 25),
        ("M17.9", "Osteoarthritis of knee", 15),
        ("S93.4", "Ankle sprain", 10),
        ("M75.4", "Shoulder impingement", 8),
        ("M25.561", "Knee pain", 12),
        ("M54.2", "Cervicalgia", 10),
    ],
    "default": [
        ("I10", "Essential hypertension", 15),
        ("E11.9", "Type 2 diabetes mellitus", 12),
        ("J06.9", "Acute upper respiratory infection", 10),
        ("M54.5", "Low back pain", 10),
        ("Z00.00", "General adult medical exam", 15),
    ]
}

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def weighted_choice(choices_with_weights):
    """Pick an item based on probability weights."""
    items, weights = zip(*[(c[:2], c[2]) if len(c) == 3 else (c, 1) for c in choices_with_weights])
    weights = [w/sum(weights) for w in weights]
    return items[np.random.choice(len(items), p=weights)]


def calculate_age(dob):
    """Calculate age from date of birth."""
    today = datetime.now().date()
    return today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))


# ============================================================================
# DATA GENERATION FUNCTIONS
# ============================================================================

def generate_patients(n=10000):
    """Generate patients with demographic distributions."""
    print(f"Generating {n} patients...")
    
    patients = []
    for patient_id in range(1, n + 1):
        # Age distribution: more adults, fewer children
        age_group = np.random.choice(
            [0, 1, 2, 3, 4],
            p=[0.05, 0.10, 0.25, 0.35, 0.25]  # 0-17, 18-30, 31-50, 51-70, 70+
        )
        
        if age_group == 0:
            min_age, max_age = 1, 17
        elif age_group == 1:
            min_age, max_age = 18, 30
        elif age_group == 2:
            min_age, max_age = 31, 50
        elif age_group == 3:
            min_age, max_age = 51, 70
        else:
            min_age, max_age = 71, 90
        
        dob = fake.date_of_birth(minimum_age=min_age, maximum_age=max_age)
        age = calculate_age(dob)
        
        # Older patients more likely to have certain conditions (will affect diagnoses later)
        provider, plan = random.choice(INSURANCE_PROVIDERS)
        
        patients.append({
            "patient_id": patient_id,
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "dob": dob,
            "gender": np.random.choice(["Male", "Female"], p=[0.48, 0.52]),
            "insurance_provider": provider,
            "insurance_plan": plan,
            "date_registered": fake.date_between(start_date="-5y", end_date="today")
        })
    
    return pd.DataFrame(patients)


def generate_providers(n=50):
    """Generate providers with specialty distribution."""
    print(f"Generating {n} providers...")
    
    providers = []
    for provider_id in range(1, n + 1):
        # More primary care docs than specialists
        specialty = np.random.choice(
            SPECIALTIES,
            p=[0.20, 0.15, 0.10, 0.08, 0.10, 0.07, 0.05, 0.08, 0.10, 0.07]
        )
        
        providers.append({
            "provider_id": provider_id,
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "specialty": specialty,
            "npi": ''.join([str(random.randint(0, 9)) for _ in range(10)])
        })
    
    return pd.DataFrame(providers)


def generate_appointments(patients_df, providers_df, n=50000):
    """Generate appointments with realistic workflow patterns."""
    print(f"Generating {n} appointments...")
    
    # Create patient-provider assignment logic
    # Pediatrics providers only see patients under 18
    ped_providers = providers_df[providers_df['specialty'] == 'Pediatrics']['provider_id'].tolist()
    other_providers = providers_df[providers_df['specialty'] != 'Pediatrics']['provider_id'].tolist()
    
    # Add age to patients for filtering
    patients_df['age'] = patients_df['dob'].apply(calculate_age)
    pediatric_patients = patients_df[patients_df['age'] < 18]['patient_id'].tolist()
    adult_patients = patients_df[patients_df['age'] >= 18]['patient_id'].tolist()
    
    appointments = []
    start_date = datetime.now() - timedelta(days=730)  # 2 years back
    
    for i in range(1, n + 1):
        # Assign patient to appropriate provider
        if random.random() < 0.15 and pediatric_patients:  # 15% pediatric visits
            patient_id = random.choice(pediatric_patients)
            provider_id = random.choice(ped_providers) if ped_providers else random.choice(other_providers)
        else:
            patient_id = random.choice(adult_patients) if adult_patients else random.choice(pediatric_patients)
            provider_id = random.choice(other_providers)
        
        # Schedule: more appointments on weekdays, 8 AM to 5 PM
        scheduled_date = fake.date_time_between(start_date=start_date, end_date=datetime.now())
        # Shift to business hours
        scheduled_date = scheduled_date.replace(
            hour=random.randint(8, 16),
            minute=random.choice([0, 15, 30, 45]),
            second=0,
            microsecond=0
        )
        
        # Status with realistic distribution
        status_roll = random.random()
        if status_roll < 0.08:
            status = "Cancelled"
            actual_start = None
            actual_end = None
        elif status_roll < 0.20:
            status = "No-Show"
            actual_start = None
            actual_end = None
        elif status_roll < 0.90:
            status = "Completed"
            # Realistic wait times: 5-45 minutes after scheduled time
            wait_minutes = random.randint(5, 45)
            actual_start = scheduled_date + timedelta(minutes=wait_minutes)
            # Visit duration: 10-60 minutes
            duration = random.randint(10, 60)
            actual_end = actual_start + timedelta(minutes=duration)
        else:
            status = "Checked-In"
            actual_start = scheduled_date + timedelta(minutes=random.randint(5, 45))
            actual_end = None
        
        appointments.append({
            "appointment_id": i,
            "patient_id": patient_id,
            "provider_id": provider_id,
            "scheduled_time": scheduled_date,
            "actual_start_time": actual_start,
            "actual_end_time": actual_end,
            "status": status
        })
        
        if i % 10000 == 0:
            print(f"  Generated {i} appointments...")
    
    return pd.DataFrame(appointments)


def generate_encounters(appointments_df, providers_df):
    """Generate encounters only for completed appointments."""
    print("Generating encounters...")
    
    # Only completed appointments get encounters
    completed_appts = appointments_df[appointments_df['status'] == 'Completed'].copy()
    
    encounters = []
    encounter_id = 1
    
    for _, appt in completed_appts.iterrows():
        # Get provider specialty for clinical context
        provider_specialty = providers_df[providers_df['provider_id'] == appt['provider_id']]['specialty'].values[0]
        
        # Age-appropriate chief complaints
        patient_id = appt['patient_id']
        # We'll map patient age later, for now use random based on specialty
        complaints_by_specialty = {
            "Cardiology": ["Chest pain", "Palpitations", "Shortness of breath", "Hypertension follow-up"],
            "Pediatrics": ["Fever", "Cough", "Ear pain", "Well-child check", "Rash"],
            "Orthopedics": ["Back pain", "Knee pain", "Shoulder pain", "Ankle injury"],
            "Family Medicine": ["Annual physical", "Cold symptoms", "Headache", "Fatigue", "Blood pressure check"],
            "Internal Medicine": ["Diabetes follow-up", "Hypertension management", "Preventive visit"],
            "default": ["Routine follow-up", "New symptoms", "Medication refill", "Preventive screening"]
        }
        
        complaints = complaints_by_specialty.get(provider_specialty, complaints_by_specialty["default"])
        
        encounters.append({
            "encounter_id": encounter_id,
            "appointment_id": appt['appointment_id'],
            "patient_id": appt['patient_id'],
            "provider_id": appt['provider_id'],
            "encounter_date": appt['actual_start_time'].date() if pd.notna(appt['actual_start_time']) else appt['scheduled_time'].date(),
            "chief_complaint": random.choice(complaints)
        })
        encounter_id += 1
        
        if encounter_id % 5000 == 0:
            print(f"  Generated {encounter_id} encounters...")
    
    return pd.DataFrame(encounters)


def generate_diagnoses(encounters_df, providers_df, patients_df):
    """Generate diagnoses based on specialty and patient age."""
    print("Generating diagnoses...")
    
    # Pre-compute patient ages
    patient_ages = dict(zip(patients_df['patient_id'], patients_df['dob'].apply(calculate_age)))
    
    diagnoses = []
    diagnosis_id = 1
    
    for _, enc in encounters_df.iterrows():
        provider_specialty = providers_df[providers_df['provider_id'] == enc['provider_id']]['specialty'].values[0]
        patient_age = patient_ages.get(enc['patient_id'], 45)
        
        # Get relevant ICD codes
        specialty_codes = ICD10_BY_SPECIALTY.get(provider_specialty, ICD10_BY_SPECIALTY["default"])
        
        # Number of diagnoses per encounter (1-4)
        num_diagnoses = np.random.choice([1, 2, 3, 4], p=[0.20, 0.40, 0.25, 0.15])
        
        used_codes = set()
        for rank in range(1, num_diagnoses + 1):
            # Pick a diagnosis not already used
            available_codes = [(c, d, w) for (c, d, w) in specialty_codes if c not in used_codes]
            if not available_codes:
                break
            
            # Weight-based selection
            codes_list = []
            weights_list = []
            for code, desc, weight in available_codes:
                codes_list.append((code, desc))
                weights_list.append(weight)
            
            weights_list = [w/sum(weights_list) for w in weights_list]
            chosen_idx = np.random.choice(len(codes_list), p=weights_list)
            code, desc = codes_list[chosen_idx]
            
            # Business rule: older patients more likely to have chronic conditions
            if patient_age > 60 and code in ["I10", "E11.9", "E78.5", "I25.10"] and rank == 1:
                pass  # Keep the chronic condition
            elif patient_age < 18 and code in ["Z00.129", "J06.9", "H66.9"]:
                pass  # Age-appropriate pediatric codes
            
            diagnoses.append({
                "diagnosis_id": diagnosis_id,
                "encounter_id": enc['encounter_id'],
                "icd_code": code,
                "description": desc,
                "diagnosis_rank": rank
            })
            
            used_codes.add(code)
            diagnosis_id += 1
    
    print(f"  Generated {diagnosis_id - 1} diagnoses")
    return pd.DataFrame(diagnoses)

def generate_procedures(encounters_df, providers_df):
    """Generate procedures based on encounter context."""
    print("Generating procedures...")
    
    # Create lookup dictionaries for quick access
    cpt_desc = {}
    cpt_charge = {}
    for code, desc, charge, _ in CPT_CODES:
        cpt_desc[code] = desc
        cpt_charge[code] = charge
    
    procedures = []
    procedure_id = 1
    
    for _, enc in encounters_df.iterrows():
        provider_specialty = providers_df[providers_df['provider_id'] == enc['provider_id']]['specialty'].values[0]
        
        # Procedure mix varies by specialty
        specialty_proc_weights = {
            "Cardiology": [("93000", 0.4), ("85025", 0.2), ("99214", 0.4)],
            "Orthopedics": [("97110", 0.3), ("97140", 0.2), ("71045", 0.3), ("99214", 0.2)],
            "Family Medicine": [("99213", 0.3), ("99214", 0.2), ("85025", 0.15), ("80053", 0.15), ("90471", 0.2)],
            "Internal Medicine": [("99214", 0.4), ("85025", 0.2), ("80053", 0.2), ("93000", 0.2)],
            "Pediatrics": [("99213", 0.3), ("90471", 0.3), ("85025", 0.2), ("81003", 0.2)],
            "default": [("99213", 0.4), ("99214", 0.3), ("85025", 0.15), ("80053", 0.15)]
        }
        
        proc_weights = specialty_proc_weights.get(provider_specialty, specialty_proc_weights["default"])
        
        # Number of procedures (1-5, weighted toward 1-2)
        num_procs = np.random.choice([1, 2, 3, 4, 5], p=[0.30, 0.30, 0.20, 0.15, 0.05])
        
        used_procs = set()
        for _ in range(num_procs):
            # Filter to unused procedures
            available = [(code, weight) for code, weight in proc_weights if code not in used_procs]
            
            if not available:
                break
            
            # Weighted random selection
            codes = [a[0] for a in available]
            weights = [a[1] for a in available]
            # Normalize weights to sum to 1
            total = sum(weights)
            weights = [w/total for w in weights]
            chosen_code = np.random.choice(codes, p=weights)
            
            # Get description and base charge
            desc = cpt_desc.get(chosen_code, "Unknown procedure")
            base_charge = cpt_charge.get(chosen_code, 75.00)
            
            # Add realistic variation to charge (±15%)
            charge = round(base_charge * random.uniform(0.85, 1.15), 2)
            
            procedures.append({
                "procedure_id": procedure_id,
                "encounter_id": enc['encounter_id'],
                "cpt_code": chosen_code,
                "description": desc,
                "units": random.randint(1, 3) if chosen_code in ["97110", "97140"] else 1,
                "charge_amount": charge
            })
            
            used_procs.add(chosen_code)
            procedure_id += 1
        
        # Progress update
        if procedure_id % 10000 == 0:
            print(f"  Generated {procedure_id} procedures...")
    
    print(f"  Total procedures generated: {procedure_id - 1}")
    return pd.DataFrame(procedures)

    """Generate procedures based on encounter context."""
    print("Generating procedures...")
    
    procedures = []
    procedure_id = 1
    
    for _, enc in encounters_df.iterrows():
        provider_specialty = providers_df[providers_df['provider_id'] == enc['provider_id']]['specialty'].values[0]
        
        # Procedure mix varies by specialty
        specialty_proc_weights = {
            "Cardiology": [("93000", 0.4), ("85025", 0.2), ("99214", 0.4)],
            "Orthopedics": [("97110", 0.3), ("97140", 0.2), ("71045", 0.3), ("99214", 0.2)],
            "Family Medicine": [("99213", 0.3), ("99214", 0.2), ("85025", 0.15), ("80053", 0.15), ("90471", 0.2)],
            "Internal Medicine": [("99214", 0.4), ("85025", 0.2), ("80053", 0.2), ("93000", 0.2)],
            "Pediatrics": [("99213", 0.3), ("90471", 0.3), ("85025", 0.2), ("81003", 0.2)],
            "default": [("99213", 0.4), ("99214", 0.3), ("85025", 0.15), ("80053", 0.15)]
        }
        
        proc_weights = specialty_proc_weights.get(provider_specialty, specialty_proc_weights["default"])
        
        # Number of procedures (1-5, weighted toward 1-2)
        num_procs = np.random.choice([1, 2, 3, 4, 5], p=[0.30, 0.30, 0.20, 0.15, 0.05])
        
        used_procs = set()
        for _ in range(num_procs):
            available = [(c, d, a, w) for (c, d, a, w) in [(code, next((desc for c, desc, _ in CPT_CODES if c == code), ""), 
                                                              next((amt for c, _, amt, _ in CPT_CODES if c == code), 75.00),
                                                              weight) 
                                                             for code, weight in proc_weights if code not in used_procs]]
            
            if not available:
                break
            
            weights = [w for _, _, _, w in available]
            weights = [w/sum(weights) for w in weights]
            chosen = available[np.random.choice(len(available), p=weights)]
            
            code, desc, base_charge, _ = chosen
            # Add some variation to charges
            charge = round(base_charge * random.uniform(0.85, 1.15), 2)
            
            procedures.append({
                "procedure_id": procedure_id,
                "encounter_id": enc['encounter_id'],
                "cpt_code": code,
                "description": desc,
                "units": random.randint(1, 3) if code in ["97110", "97140"] else 1,
                "charge_amount": charge
            })
            
            used_procs.add(code)
            procedure_id += 1
    
    print(f"  Generated {procedure_id - 1} procedures")
    return pd.DataFrame(procedures)


def generate_claims_and_line_items(encounters_df, procedures_df):
    """Generate claims with realistic financial outcomes."""
    print("Generating claims and line items...")
    
    claims = []
    line_items = []
    claim_id = 1
    line_id = 1
    
    for _, enc in encounters_df.iterrows():
        enc_procs = procedures_df[procedures_df['encounter_id'] == enc['encounter_id']]
        
        if enc_procs.empty:
            continue
        
        total_charge = enc_procs['charge_amount'].sum()
        filing_date = enc['encounter_date'] + timedelta(days=random.randint(1, 7))
        
        # Claim status with realistic distribution
        status_roll = random.random()
        if status_roll < 0.10:
            status = "Denied"
            adjudication_date = filing_date + timedelta(days=random.randint(14, 45))
        elif status_roll < 0.20:
            status = "Pending"
            adjudication_date = None
        elif status_roll < 0.75:
            status = "Paid"
            adjudication_date = filing_date + timedelta(days=random.randint(10, 60))
        else:
            status = "Partial"
            adjudication_date = filing_date + timedelta(days=random.randint(20, 60))
        
        claims.append({
            "claim_id": claim_id,
            "encounter_id": enc['encounter_id'],
            "patient_id": enc['patient_id'],
            "provider_id": enc['provider_id'],
            "total_charge": round(total_charge, 2),
            "claim_status": status,
            "filing_date": filing_date,
            "adjudication_date": adjudication_date
        })
        
        # Generate line items for each procedure on this claim
        for _, proc in enc_procs.iterrows():
            charge = proc['charge_amount'] * proc['units']
            
            if status == "Paid":
                # Insurance typically pays 60-90% of charges
                payment_rate = random.uniform(0.60, 0.90)
                payment = round(charge * payment_rate, 2)
                adjustment = round(charge - payment, 2)
            elif status == "Denied":
                # Denied: no payment, full adjustment (write-off)
                payment = 0.00
                adjustment = round(charge, 2)
            elif status == "Partial":
                # Partial payment
                payment = round(charge * random.uniform(0.30, 0.70), 2)
                adjustment = round(charge - payment, 2)
            else:  # Pending
                payment = 0.00
                adjustment = 0.00
            
            line_items.append({
                "line_id": line_id,
                "claim_id": claim_id,
                "procedure_id": proc['procedure_id'],
                "charge_amount": round(charge, 2),
                "payment_amount": payment,
                "adjustment_amount": adjustment
            })
            line_id += 1
        
        claim_id += 1
        if claim_id % 5000 == 0:
            print(f"  Generated {claim_id} claims...")
    
    print(f"  Generated {claim_id - 1} claims and {line_id - 1} line items")
    return pd.DataFrame(claims), pd.DataFrame(line_items)


def generate_payments(claims_df):
    """Generate payments only for paid/partial claims."""
    print("Generating payments...")
    
    # Only paid or partially paid claims get payments
    eligible_claims = claims_df[claims_df['claim_status'].isin(['Paid', 'Partial'])]
    
    payments = []
    payment_id = 1
    
    for _, claim in eligible_claims.iterrows():
        # Insurance payment (80% of cases) or patient payment (20%)
        payment_type = np.random.choice(["Insurance", "Patient"], p=[0.80, 0.20])
        
        # Payment amount (could be split into multiple payments)
        num_payments = np.random.choice([1, 1, 1, 2], p=[0.70, 0.15, 0.10, 0.05])  # Usually 1 payment
        
        if claim['claim_status'] == 'Paid':
            total_payment = claim['total_charge'] * random.uniform(0.75, 0.95)
        else:  # Partial
            total_payment = claim['total_charge'] * random.uniform(0.30, 0.70)
        
        # Split into multiple payments if needed
        for i in range(num_payments):
            if i == num_payments - 1:
                amount = round(total_payment - sum(p['amount'] for p in payments if p['claim_id'] == claim['claim_id']), 2)
            else:
                amount = round(total_payment * random.uniform(0.3, 0.7), 2)
            
            payment_date = claim['adjudication_date'] + timedelta(days=random.randint(5, 30)) if pd.notna(claim['adjudication_date']) else claim['filing_date'] + timedelta(days=45)
            
            payments.append({
                "payment_id": payment_id,
                "claim_id": claim['claim_id'],
                "payment_date": payment_date,
                "amount": max(amount, 0.01),
                "payment_type": payment_type
            })
            payment_id += 1
    
    print(f"  Generated {payment_id - 1} payments")
    return pd.DataFrame(payments)


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main(load_to_db=True, save_csv=True):
    """Generate all data and optionally load to database."""
    
    print("\n" + "="*60)
    print("MEDICAL PRACTICE SYNTHETIC DATA GENERATOR")
    print("="*60 + "\n")
    
    # Generate reference data
    patients_df = generate_patients(10000)
    providers_df = generate_providers(50)
    
    # Generate workflow data
    appointments_df = generate_appointments(patients_df, providers_df, 50000)
    
    # Clinical data
    encounters_df = generate_encounters(appointments_df, providers_df)
    diagnoses_df = generate_diagnoses(encounters_df, providers_df, patients_df)
    procedures_df = generate_procedures(encounters_df, providers_df)
    
    # Financial data
    claims_df, line_items_df = generate_claims_and_line_items(encounters_df, procedures_df)
    payments_df = generate_payments(claims_df)
    
    # ============================================================================
    # DATA QUALITY VALIDATION
    # ============================================================================
    
    print("\n" + "="*60)
    print("DATA QUALITY CHECKS")
    print("="*60 + "\n")
    
    # Rule 1: Cancelled appointments have no encounter
    cancelled_appts = appointments_df[appointments_df['status'] == 'Cancelled']['appointment_id']
    cancelled_with_encounters = encounters_df[encounters_df['appointment_id'].isin(cancelled_appts)]
    print(f"✓ Cancelled appointments with encounters (should be 0): {len(cancelled_with_encounters)}")
    
    # Rule 2: Only completed encounters generate claims
    completed_appts = appointments_df[appointments_df['status'] == 'Completed']['appointment_id']
    encounter_ids = set(encounters_df[encounters_df['appointment_id'].isin(completed_appts)]['encounter_id'])
    claim_encounter_ids = set(claims_df['encounter_id'])
    claims_from_non_encounters = claim_encounter_ids - encounter_ids
    print(f"✓ Claims from non-encounters (should be 0): {len(claims_from_non_encounters)}")
    
    # Rule 3: Denied claims have $0 payment
    denied_claim_ids = claims_df[claims_df['claim_status'] == 'Denied']['claim_id']
    denied_line_items = line_items_df[line_items_df['claim_id'].isin(denied_claim_ids)]
    denied_with_payment = denied_line_items[denied_line_items['payment_amount'] > 0]
    print(f"✓ Denied claims with payments (should be 0): {len(denied_with_payment)}")
    
    # Rule 4: Payment totals match
    total_claims_charged = claims_df['total_charge'].sum()
    total_line_charged = line_items_df['charge_amount'].sum()
    print(f"✓ Charge reconciliation: Claims=${total_claims_charged:,.2f}, Line items=${total_line_charged:,.2f}")
    
    # Dataset size summary
    print("\n" + "="*60)
    print("DATASET SIZES")
    print("="*60)
    print(f"Patients:       {len(patients_df):,}")
    print(f"Providers:      {len(providers_df):,}")
    print(f"Appointments:   {len(appointments_df):,}")
    print(f"Encounters:     {len(encounters_df):,}")
    print(f"Diagnoses:      {len(diagnoses_df):,}")
    print(f"Procedures:     {len(procedures_df):,}")
    print(f"Claims:         {len(claims_df):,}")
    print(f"Line Items:     {len(line_items_df):,}")
    print(f"Payments:       {len(payments_df):,}")
    
    # Save to CSV if requested
    if save_csv:
        print("\n" + "="*60)
        print("SAVING TO CSV")
        print("="*60)
        
        tables = {
            "patients": patients_df,
            "providers": providers_df,
            "appointments": appointments_df,
            "encounters": encounters_df,
            "diagnoses": diagnoses_df,
            "procedures": procedures_df,
            "claims": claims_df,
            "claim_line_items": line_items_df,
            "payments": payments_df
        }
        
        for name, df in tables.items():
            path = f"../../data/synthetic/{name}.csv"
            df.to_csv(path, index=False)
            print(f"  Saved {path} ({len(df):,} rows)")
    
    # Load to PostgreSQL if requested
    if load_to_db:
        print("\n" + "="*60)
        print("LOADING TO POSTGRESQL")
        print("="*60)
        
        table_order = [
            ("patients", patients_df),
            ("providers", providers_df),
            ("appointments", appointments_df),
            ("encounters", encounters_df),
            ("diagnoses", diagnoses_df),
            ("procedures", procedures_df),
            ("claims", claims_df),
            ("claim_line_items", line_items_df),
            ("payments", payments_df)
        ]
        
        for table_name, df in table_order:
            try:
                df.to_sql(table_name, engine, if_exists="replace", index=False, method="multi", chunksize=1000)
                print(f"  ✓ Loaded {table_name}: {len(df):,} rows")
            except Exception as e:
                print(f"  ✗ Error loading {table_name}: {e}")
        
        # Create indexes
        print("\n  Creating indexes...")
        index_sql = """
        CREATE INDEX IF NOT EXISTS idx_appointments_patient ON appointments(patient_id);
        CREATE INDEX IF NOT EXISTS idx_appointments_provider ON appointments(provider_id);
        CREATE INDEX IF NOT EXISTS idx_appointments_status ON appointments(status);
        CREATE INDEX IF NOT EXISTS idx_encounters_appointment ON encounters(appointment_id);
        CREATE INDEX IF NOT EXISTS idx_diagnoses_encounter ON diagnoses(encounter_id);
        CREATE INDEX IF NOT EXISTS idx_procedures_encounter ON procedures(encounter_id);
        CREATE INDEX IF NOT EXISTS idx_claims_encounter ON claims(encounter_id);
        CREATE INDEX IF NOT EXISTS idx_claims_status ON claims(claim_status);
        CREATE INDEX IF NOT EXISTS idx_line_items_claim ON claim_line_items(claim_id);
        CREATE INDEX IF NOT EXISTS idx_payments_claim ON payments(claim_id);
        """
        
        try:
            with engine.connect() as conn:
                for statement in index_sql.strip().split(';'):
                    if statement.strip():
                        conn.execute(statement)
                conn.commit()
            print("  ✓ Indexes created successfully")
        except Exception as e:
            print(f"  ✗ Error creating indexes: {e}")
    
    print("\n" + "="*60)
    print("GENERATION COMPLETE!")
    print("="*60)
    
    return {
        "patients": patients_df,
        "providers": providers_df,
        "appointments": appointments_df,
        "encounters": encounters_df,
        "diagnoses": diagnoses_df,
        "procedures": procedures_df,
        "claims": claims_df,
        "line_items": line_items_df,
        "payments": payments_df
    }


if __name__ == "__main__":
    data = main(load_to_db=False, save_csv=True)


MEDICAL PRACTICE SYNTHETIC DATA GENERATOR

Generating 10000 patients...
Generating 50 providers...
Generating 50000 appointments...
  Generated 10000 appointments...
  Generated 20000 appointments...
  Generated 30000 appointments...
  Generated 40000 appointments...
  Generated 50000 appointments...
Generating encounters...
  Generated 5000 encounters...
  Generated 10000 encounters...
  Generated 15000 encounters...
  Generated 20000 encounters...
  Generated 25000 encounters...
  Generated 30000 encounters...
Generating diagnoses...
  Generated 81748 diagnoses
Generating procedures...
  Generated 10000 procedures...
  Generated 40000 procedures...
  Generated 50000 procedures...
  Generated 70000 procedures...
  Total procedures generated: 80613
Generating claims and line items...
  Generated 5000 claims...
  Generated 10000 claims...
  Generated 15000 claims...
  Generated 20000 claims...
  Generated 25000 claims...
  Generated 30000 claims...
  Generated 34840 claims and 80613 li